# IBSpyHelper - pangenome-informed haplotype pipeline for a target interval

This notebook runs the whole IBSpyHelper analysis for **one target interval**, from raw IBSpy
affinity-propagation (AP) output to the haplotype-sharing summaries and plots. It combines two
previously separate notebooks into one ordered pipeline:

| Part | What it does | Helper script |
|------|--------------|---------------|
| 1 | Per-reference extraction: haplotypes, unique haplogroups and their member genotypes | `IBSpyHelper_v2.py` |
| 2 | Collate the haplogroup of every genotype across all references into one table | `IBSpyHelperSummaries.py` |
| 3 | For a genotype of interest, find the genotypes that share its haplotype | `IBSpyHelperSummaries.py` |

### The concept

IBSpy counts k-mer variations between each **query** (genotype in the panel) and each **reference**
(pangenome assembly), and affinity propagation groups the queries into haplotypes block by block
along the chromosome. For a target interval, a query's string of per-block haplotype calls *is* its
haplotype at that locus; queries with an identical string belong to the same **haplogroup**.

Because the interval is called against ten different assemblies, the same panel is haplotyped ten
times over (using the syntenic coordinates of the interval in each assembly). A haplotype that is a
genuine shared ancestral segment keeps the same set of genotypes together in **all** references; one
that is an artefact of a single assembly does not. Parts 3 turn that into a number: for a given
genotype, in how many of the ten references does another genotype fall in the same haplogroup.

### Inputs

- One IBSpy AP output file per reference (`*.tsv.gz`, tab-separated) in `<region>/raw-data-files/`.
- The two helper scripts, `IBSpyHelper_v2.py` and `IBSpyHelperSummaries.py`, which are copied into
  the working directory before being imported.

### Outputs (sub-directories of `<region>/`)

`variation-files/` · `haplotypes/` · `hapID_haplotype_queryList_files/` ·
`haplotype_with_hapGroupId_files/` · `haplotype_with_hapGroupId_files/genotype_frequency_tables/`

### Before you run — the cells to edit

1. **Part 1, cell 1** - project directory (`os.chdir`).
2. **Part 1, cell 2** - `region`, the target-interval folder name.
3. **Part 1, cell 4** and **Part 2, cell 2** - absolute path of the two helper scripts to copy in.
4. **Part 1, cell 7** - `ref_up_down_dict`: the syntenic start/end of the interval in every
   reference (obtained from the `extractSynteny` script).
5. **Part 2, cell 1** - path to re-enter the region directory if you start the notebook at Part 2.
6. **Part 2, cell 5** - `references` and `base_ref`.
7. **Part 3, cell 1** - `genotype_to_search` and `ref_threshold`.

Run the parts in order on a fresh kernel.

---
## Part 1 - Per-reference haplotype extraction

The interval is `2D: 635599182..650840934`, run here as **634-645 Mb on Chinese Spring** with the
syntenic interval in each of the other assemblies. Coordinates come from the SNP and SSR markers;
the syntenic intervals were identified with the `extractSynteny` script.

For every reference the loop extracts the rows of the interval, writes the variation table, builds
the block × query haplotype table, names the unique haplotypes and lists their member genotypes, and
finally writes the haplotype table with each query's haplogroup ID attached - the input to Part 2.

Imports, and the project directory that holds the target-region folders.

In [ ]:
# import reqiured modules
import os
import glob
import csv
from datetime import date
## set working directory

os.chdir("/Users/waweru/Documents/2022/Python_learning/python_learning_with_Burkhard/2026-Aug-18_Rerun_CS_634-645_ForThesis/")




Create the folder for the target region if it does not exist. The D-genome interval is run
against the groups that include the *Ae. tauschii* accessions.

In [ ]:
region = 'chr2D_CS_634-645Mb'

isExist = os.path.exists(f"{os.getcwd()}/{region}")

if isExist:
    print('directory', region, 'exists')
else:
    # Create a new directory because it does not exist
    os.makedirs(f"{os.getcwd()}/{region}")
    print("The new directory is created!")


Move into it - everything from here on is written relative to this directory.

In [ ]:
work_dir = f"{os.getcwd()}/{region}"

os.chdir(work_dir)

print(work_dir)

Copy the helper script into the working directory, so the run is self-documenting, then import it.

In [ ]:
import shutil

source_file = "/Users/waweru/Documents/2022/Python_learning/python_learning_with_Burkhard/IBSpyHelper_v2.py"
desti_dir = f"{work_dir}"

shutil.copy(source_file, desti_dir)

In [ ]:
import IBSpyHelper_v2

dir(IBSpyHelper_v2)

Create the output directories for the intermediate steps.

In [ ]:
# most 
import os
folders = ['raw-data-files','variation-files', 'haplotypes','hapID_haplotype_queryList_files', 'haplotype_with_hapGroupId_files'] # 'raw-data-files',
# Check whether the specified path exists or not
for folder in folders:
    isExist = os.path.exists(f"{os.getcwd()}/{folder}")
    if isExist:
        print('directory', folder, 'exists')
    else:
        # Create a new directory because it does not exist
        os.makedirs(f"{os.getcwd()}/{folder}")
        print("The new directory is created!")

Copy the raw IBSpy AP files into `raw-data-files/` now, then check they are all there. Each
file name must contain the reference name used as a key in the next cell, which is how the loop
pairs a file with its interval coordinates.

In [ ]:
import glob
glob.glob(f"{os.getcwd()}/raw-data-files/*.tsv.gz")

**The main loop.** `ref_up_down_dict` holds the syntenic start and end of the target interval
in each reference. For each reference in turn:

- `getTargetRows` - pull the rows of the interval, returning the variation table and the per-block
  haplotype calls; the variation table is written to `variation-files/`
- `getHapDf` - pivot the calls into a block × query haplotype table → `haplotypes/`
- `getUniqHapQueries` - collapse to unique haplotypes, name them, count and list their genotypes
- `SaveHapsQeueriesToTsv` - write that to `hapID_haplotype_queryList_files/`
- `AddHaploGroup` - attach each query's haplogroup ID → `haplotype_with_hapGroupId_files/`

All output names carry the reference, the interval and the run date.

In [ ]:
import csv
from datetime import date
import pandas as pd

#define the references and their start and end regions
rawdata = f"{os.getcwd()}/raw-data-files/"

ref_up_down_dict = {'chinese': {'start': 634_000_000, 'end': 645_000_000},
                   'arinalrfor': {'start': 639_000_000, 'end': 650_000_000},
                   'jagger': {'start': 658_000_000, 'end': 666_000_000},
                   'mace': {'start': 632_000_000, 'end': 637_000_000},
                   'norin61': {'start': 631_000000, 'end': 650_000_000},
                   'julius': {'start': 648_000_000, 'end': 655_000_000},
                   'lancer': {'start': 631_000_000, 'end': 633_000_000},
                   'stanley': {'start': 637_000_000, 'end': 647_000_000},
                   'landmark': {'start': 638_000_000, 'end': 644_000_000},
                   'sy_mattis': {'start': 637_000_000, 'end': 644_000_000}}

# for loop to use the reference, the interval start and end for each

for reference, inner_dict in ref_up_down_dict.items():
    #print(f'{reference}')
    for key, value in inner_dict.items():
        if key == 'start':
            intervalstart = value
        else:
            intervalend = value
    print(f'{reference}:{intervalstart}-{intervalend}')

    # next define the input raw data file name, 

    for file in os.listdir(f'{rawdata}'):
        if file.__contains__(reference):
            #print(file)
            filename=f'{rawdata}/{file}'
            print(filename)

    # next extract the variation and haps file for each reference
    print(f'now extracting variations and haplotype data for {reference} from {intervalstart} to {intervalend} Mbp')

    targetdat,haps_all = IBSpyHelper_v2.getTargetRows(filename, intervalstart, intervalend) 


    # save the variation data into a CSV file with today's date


    date = date.today()

    with open(f"variation-files/{date}_{filename.split('/')[filename.count('/')].split('.')[0]}_{str(intervalstart).replace('000000','')}-{str(intervalend).replace('000000','')}Mb_variations_haplotypes.csv", "w") as csvout:
        writer = csv.writer(csvout, delimiter="\t")
        # write the header line first
        header = ["reference,chr,block,window,query,variations,group"]
        writer.writerow(header)
        writer.writerows(targetdat)

    # next generate a properly formatted dataframe of the haplotypes 

    haps_per_block_df = IBSpyHelper_v2.getHapDf(haps_all, drop_queries=False, queries_to_drop=None)


    #then save them into a csv file

    haps_per_block_df.to_csv(f"haplotypes/{date}_{filename.split('/')[filename.count('/')].split('.')[0]}_{str(intervalstart).replace('000000','')}-{str(intervalend).replace('000000','')}Mb_haplotypes.csv")

    print(f'variation and haplotype data for {reference} saved to csv files')

    # next to get the unique haplotypes and their numbers and members 

    print(f'extracting unique haplotypes number of queries and members list for {reference} ')
    
    haplotype_file = haps_per_block_df

    hap_hapnum_queries_dict, hap_dict,query_list_dict = IBSpyHelper_v2.getUniqHapQueries(haplotype_file, filename, intervalstart,intervalend)


    # and save to TSV file

    IBSpyHelper_v2.SaveHapsQeueriesToTsv(hap_hapnum_queries_dict, filename, intervalstart, intervalend,date)

    print(f'saved hap_hapnum_queries_dict for {reference} to TSV file')

    # add the haplogroups ID of each query to the haplotype dataframe
    print(f'adding hapGroupID to each query in hapdf for {reference}')

    haps_per_block_df=haps_per_block_df
    query_list_dict=query_list_dict

    haps_per_block_df = IBSpyHelper_v2.AddHaploGroup(haps_per_block_df,query_list_dict)

    # save to csv file

    haps_per_block_df.to_csv(f"haplotype_with_hapGroupId_files/{date}_{filename.split('/')[filename.count('/')].split('.')[0]}_{str(intervalstart).replace('000000','')}-{str(intervalend).replace('000000','')}Mb_HapGroupID.csv")

    print(f'saved haps_df with hapID for each query for {reference} to TSV file')


Sanity check on the last reference processed: the variation rows, and the haplotype table with
the `genotype` and `hapgroup` columns added.

In [ ]:
targetdat[0:5]

In [ ]:
haps_per_block_df.head(n=10)

---
## Part 2 - Collate haplogroups across all references

Part 1 wrote one `*_HapGroupID.csv` per reference. Each says which haplogroup every genotype falls
into *when the interval is viewed through that assembly*. Collating them gives one table, `base_df`,
with a row per genotype and a column per reference - the basis for every summary that follows.

Chinese Spring is used as the base reference because the interval was located on CS first and the
syntenic intervals in the other assemblies were derived from it; any reference can take that role by
changing `base_ref`.

Only needed if you are starting the notebook here rather than running on from Part 1 - it
re-enters the target-region directory.

In [ ]:
import os
# change to correct directory for the target region currently working on

os.chdir("/Users/waweru/Documents/2022/Python_learning/python_learning_with_Burkhard/2026-Aug-18_Rerun_CS_634-645_ForThesis/chr2D_CS_634-645Mb")

Copy in the second helper script, which holds the summary functions, and import it.

In [ ]:
import shutil

source_file = "/Users/waweru/Documents/2022/Python_learning/python_learning_with_Burkhard/IBSpyHelperSummaries.py"
desti_dir = f"{os.getcwd()}"

shutil.copy(source_file, desti_dir)

In [ ]:
import IBSpyHelperSummaries

dir(IBSpyHelperSummaries)

The directory holding the per-reference haplogroup files from Part 1.

In [ ]:
hapgroupID_files_dir=f'{os.getcwd()}/haplotype_with_hapGroupId_files/'
hapgroupID_files_dir

Check that a file was found for every reference (`collateHapsAllRefs` also checks this, and stops if one is missing).

In [ ]:
import glob

references=['arinalrfor','chinese','jagger','julius','lancer', 'landmark','mace','norin61','stanley','sy_mattis']


# putting it to a function to read file one by one

for ref in references:
    reference_file=glob.glob(f"{hapgroupID_files_dir}/*{ref}*_HapGroupID.csv")
    print(reference_file)

Build the collated table. `collateHapsAllRefs` reads the base reference's file, then maps each
other reference's haplogroup onto the genotype column. Note it changes the working directory to
`hapgroupID_files_dir` and removes `base_ref` from the `references` list, so re-run this cell as a
whole rather than calling the function twice.

In [ ]:
import pandas as pd

references=['arinalrfor','chinese','jagger','julius','lancer', 'landmark','mace','norin61','stanley','sy_mattis']
base_ref='chinese'

base_df =IBSpyHelperSummaries.collateHapsAllRefs(hapgroupID_files_dir, references,base_ref)

base_df.head(n=10)

Save the collated table, with the region in the file name.

In [ ]:
from datetime import date

date = date.today()

region = 'chr2D_CS_634-645Mb'

base_df.to_csv(f"{hapgroupID_files_dir}/{date}_{region}_collated_hapGroups.csv")
base_df.to_excel(f"{hapgroupID_files_dir}/{date}_{region}_collated_hapGroups.xlsx")

---
## Part 3 - Genotypes that share the haplotype of a genotype of interest

Entering from one genotype - typically a donor or a line carrying the resistance of interest -
`getQueryHapRelatedGenotypes` asks, reference by reference, which other genotypes sit in the same
haplogroup. Each reference contributes a `<reference>_freq` column scored 1 (same haplogroup) or 0,
`freq_total` sums them, and `ref_threshold` keeps only genotypes that match in at least that many
references. Columns are ordered by column sum, so the references where the haplotype is most widely
shared group together.

`freq_total = 10` means the two genotypes are grouped together no matter which assembly the interval
is read against: the strongest evidence for a genuinely shared segment.

This function adds the `*_freq` columns to `base_df` in place, so the columns you see there reflect
whichever genotype was searched last.

Set the genotype to enter from and the minimum number of references, then build the table.

In [ ]:

base_df = base_df
references=['arinalrfor','chinese','jagger','julius','lancer','landmark','mace','norin61','stanley','sy_mattis']
ref_threshold = 1

# the name of the query you want as the entry point, with haplotype or trait of interest
genotype_to_search = 'Alcedo'


frequency_df_filt =IBSpyHelperSummaries.getQueryHapRelatedGenotypes(base_df,genotype_to_search,references, ref_threshold)

frequency_df_filt

Save it under the genotype's name, in its own folder.

In [ ]:
# save to csv so we can read in for plotting heatmaps
from datetime import date

date = date.today()

region = 'chr2D_CS_634-645Mb'

# the directory to save the genotype frequency files

os.makedirs(f"{os.getcwd()}/genotype_frequency_tables", exist_ok=True)


frequency_df_filt.to_csv(f"{hapgroupID_files_dir}/genotype_frequency_tables/{date}_{genotype_to_search}_{region}_region_collated_frequency_table.csv")

The genotype list from that table is what gets passed to the heatmap plots.

In [ ]:
alcedo_plot_queries = frequency_df_filt['genotype'].to_list()

alcedo_plot_queries